# AI vs traditional strategies

A language-model agent and the library's reference strategies (buy-and-hold, random, momentum, mean reversion and
the Oracle) trade the same seeded market for 20 days, and `tf.evaluate` scores all of them on it.

The experiment is `08-claude-agent.py`. This notebook loads that script and runs it, so the script is the source of
truth and nothing here computes on its own. It needs `pip install "tradefloor[claude]"` for the SDK and pydantic
the script imports.

By default the Claude agent is a replay of a recorded run (`tests/fixtures/claude/example-08.json`), so it needs no
key, makes no call and costs nothing. Setting `TRADEFLOOR_LIVE_EXAMPLES=1` with `ANTHROPIC_API_KEY` makes the
script call Claude instead: one call per simulated day, 20 calls. The dollar cost depends on the model and the
account and is not stated here.

In [1]:
import importlib.util
import sys
from pathlib import Path

# The file name starts with a digit, so it is loaded by path. It is put in
# sys.modules first, as an import would, because its pydantic models resolve
# their annotations through it.
spec = importlib.util.spec_from_file_location("claude_agent", Path("08-claude-agent.py"))
example = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = example
spec.loader.exec_module(example)

from tradefloor.integrations.common import Transcript

recording = Transcript.load(example.fixture_path())
meta = recording.meta
print(f"seed {example.SEED}, {example.DAYS} days")
print(f"recording: {len(recording)} answers from {meta.get('model')}, recorded {meta.get('recorded_utc', 'date not recorded')}, preset {meta.get('model_preset')}")

seed 2026, 20 days
recording: 20 answers from claude-opus-5-5, recorded 2026-11-02T19:28:15+00:00, preset pt-v22


The table that follows is the script's own output. `pnl` and `impact` come from the scorecard. The why-right
columns score the factor each agent named as the day's driver against the engine's own attribution, beside the
score a constant answer would get, so `edge` is what reading the market added. Only the Claude agent names a
factor.

In [2]:
example.main([])

Replaying 20 answers claude-opus-5-5 gave on 2026-11-02T19:28:15+00:00, from tests/fixtures/claude/example-08.json.
No call is made. TRADEFLOOR_LIVE_EXAMPLES=1 with a key runs it live.



agent                     pnl    impact  why-right  constant    edge
--------------------------------------------------------------------
buy_and_hold            34152       0.1          -         -       -
oracle                  24656       0.0       100%       95%     +5pt
claude                      0       0.0        85%       95%    -10pt
mean_reversion         -55081       0.7          -         -       -
momentum               -73003       0.4          -         -       -
random                 -84020       0.5          -         -       -

P&L over buy-and-hold:
  random                -118172
  momentum              -107155
  mean_reversion         -89233
  oracle                  -9496
  claude                 -34152

No capture ratio on pt-v22. As on pt-v20 and pt-v21, market moves there mostly stick: each shock moves fair value for good, so even perfect knowledge of the model's fair value leaves little edge. The Oracle made money in 6 of 8 test markets and took buy-and-hol

One seed ranks the seed, not the agents, as the script says at the end. `tf.rank` repeats the comparison across
seeds with an exact paired sign test, which `02-evaluating-a-strategy.ipynb` shows for rule-based strategies.